# Explainable AI Portfolio
## LIME · SHAP · Partial Dependence Plots

This notebook walks through three core XAI techniques applied to the **UCI Breast Cancer Wisconsin** dataset.

| Notebook section | Technique | Scope |
|---|---|---|
| 1 | Setup & Model Training | — |
| 2 | LIME | Local |
| 3 | SHAP | Local + Global |
| 4 | PDP | Global |
| 5 | Side-by-side comparison | — |

---
## 1. Setup & Model Training

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

import lime
import lime.lime_tabular
import shap

# Load data
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
target_names  = data.target_names   # ['malignant', 'benign']

print(f"Dataset shape : {X.shape}")
print(f"Class balance : {np.bincount(y)}  ({target_names})")

In [ ]:
# Scale & split
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Train Gradient Boosting model
model = GradientBoostingClassifier(
    n_estimators=150,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)
model.fit(X_train, y_train)

print(classification_report(y_test, model.predict(X_test), target_names=target_names))

---
## 2. LIME — Local Interpretable Model-agnostic Explanations

**How it works:**
1. Select a data point to explain
2. Generate ~1,000 perturbed neighbours by randomly masking features
3. Get the model's prediction on each neighbour
4. Fit a **weighted linear model** — samples closer to the original point get higher weight
5. The linear model's coefficients are the LIME explanation

**Key parameters:**
- `num_features` — how many features to show in the explanation
- `num_samples` — number of perturbed instances (more = more stable, slower)

In [ ]:
# Build LIME explainer
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    X_train,
    feature_names=feature_names,
    class_names=target_names,
    mode='classification',
    random_state=42
)

# Explain a test instance
idx = 5
exp = explainer_lime.explain_instance(
    X_test[idx],
    model.predict_proba,
    num_features=10,
    num_samples=1000
)

probs = model.predict_proba(X_test[idx].reshape(1, -1))[0]
true_label = target_names[y_test[idx]]
print(f"True label     : {true_label}")
print(f"P(malignant)   : {probs[0]:.3f}")
print(f"P(benign)      : {probs[1]:.3f}")

In [ ]:
# Visualise LIME explanation
fig = exp.as_pyplot_figure(label=1)  # label=1 is 'benign'
fig.set_size_inches(9, 5)
plt.title(f"LIME Explanation  |  True: {true_label}  |  P(benign)={probs[1]:.2f}", pad=12)
plt.tight_layout()
plt.show()

# Raw weights
print("\nFeature weights:")
for feature, weight in sorted(exp.as_list(), key=lambda x: abs(x[1]), reverse=True):
    direction = '→ benign   ' if weight > 0 else '→ malignant'
    print(f"  {direction}  {weight:+.4f}  |  {feature}")

In [ ]:
# Stability check — how consistent are LIME explanations across runs?
runs = []
for seed in range(5):
    e = lime.lime_tabular.LimeTabularExplainer(
        X_train, feature_names=feature_names,
        class_names=target_names, mode='classification', random_state=seed
    ).explain_instance(X_test[idx], model.predict_proba, num_features=5)
    runs.append(dict(e.as_list()))

df_stability = pd.DataFrame(runs).T
df_stability.columns = [f'run_{i}' for i in range(5)]
print("LIME weight variance across 5 random seeds:")
print(df_stability.round(4))

---
## 3. SHAP — SHapley Additive exPlanations

**How it works:**
- Rooted in **cooperative game theory** (Shapley 1953)
- Each feature's value is treated as a 'player'; the model output is the 'payout'
- The Shapley value = the feature's **average marginal contribution** across all possible orderings
- `TreeExplainer` computes exact values in polynomial time for tree models

**Additive property:** `prediction = base_value + sum(SHAP values)`

In [ ]:
# Compute SHAP values for entire test set
explainer_shap = shap.TreeExplainer(model)
shap_values = explainer_shap.shap_values(X_test)

# shap_values is a list of 2 arrays [class_0, class_1]
sv = shap_values[1]  # benign class

print(f"SHAP values shape : {sv.shape}")
print(f"Base value (benign): {explainer_shap.expected_value[1]:.4f}")

# Verify additivity for one instance
pred_log_odds = explainer_shap.expected_value[1] + sv[idx].sum()
print(f"\nAdditivity check (instance {idx}):")
print(f"  base + sum(SHAP) = {pred_log_odds:.4f}")
print(f"  model raw output = {model.decision_function(X_test[idx].reshape(1,-1))[0]:.4f}")

In [ ]:
# Global summary — beeswarm plot
shap.summary_plot(sv, X_test, feature_names=feature_names, show=True)

In [ ]:
# Global summary — mean |SHAP| bar chart
shap.summary_plot(sv, X_test, feature_names=feature_names,
                  plot_type='bar', show=True)

In [ ]:
# Local explanation — waterfall plot for one instance
shap.waterfall_plot(
    shap.Explanation(
        values=sv[idx],
        base_values=explainer_shap.expected_value[1],
        data=X_test[idx],
        feature_names=list(feature_names),
    )
)

In [ ]:
# Dependence plot — relationship between a feature and its SHAP value
top_feature_idx = np.argsort(np.abs(sv).mean(0))[-1]
print(f"Most important feature: {feature_names[top_feature_idx]}")

shap.dependence_plot(
    top_feature_idx,
    sv,
    X_test,
    feature_names=feature_names,
    show=True
)

---
## 4. PDP — Partial Dependence Plots

**How it works:**
1. Choose a feature of interest
2. Create a grid of values for that feature
3. For each grid value: copy the entire dataset, set the feature to that value, average the predictions
4. Plot the average prediction vs. feature value

**Variants:**
- `kind='average'` → classic PDP
- `kind='individual'` → ICE (Individual Conditional Expectation)
- `kind='both'` → ICE + PDP overlay

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# Top 4 features by SHAP importance
feat_importance = np.abs(sv).mean(0)
top4 = np.argsort(feat_importance)[-4:][::-1].tolist()
print("Top 4 features:", [feature_names[i] for i in top4])

# Classic PDP
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
PartialDependenceDisplay.from_estimator(
    model,
    X_train,
    features=top4,
    feature_names=feature_names,
    kind='average',
    ax=axes,
    line_kw={'color': '#58a6ff', 'linewidth': 2},
)
fig.suptitle('Partial Dependence Plots — Top 4 Features', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ICE plot — shows heterogeneous effects across individuals
fig, ax = plt.subplots(figsize=(8, 5))
PartialDependenceDisplay.from_estimator(
    model,
    X_train,
    features=[top4[0]],
    feature_names=feature_names,
    kind='both',          # ICE lines + bold PDP average
    ax=ax,
    ice_lines_kw={'color': '#8b949e', 'linewidth': 0.4, 'alpha': 0.4},
    pd_line_kw={'color': '#f78166', 'linewidth': 2.5},
    centered=True,        # center ICE at leftmost point for easier comparison
)
ax.set_title(f'ICE + PDP  —  {feature_names[top4[0]]}', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 2-D interaction PDP heatmap
fig, ax = plt.subplots(figsize=(6, 5))
PartialDependenceDisplay.from_estimator(
    model,
    X_train,
    features=[(top4[0], top4[1])],  # tuple = 2-D
    feature_names=feature_names,
    ax=ax,
)
ax.set_title(f'2-D PDP Interaction\n{feature_names[top4[0]]} × {feature_names[top4[1]]}', fontsize=11)
plt.tight_layout()
plt.show()

---
## 5. Side-by-side Comparison

For the same test instance, compare what each method says about its prediction.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    f'Three Lenses on Instance #{idx}  |  True: {true_label}  |  P(benign)={probs[1]:.2f}',
    fontsize=13, y=1.02
)

# — LIME (left) —
items = sorted(exp.as_list(), key=lambda x: x[1])
vals  = [v for _, v in items]
labs  = [l for l, _ in items]
colors = ['#3fb950' if v > 0 else '#f78166' for v in vals]
axes[0].barh(labs, vals, color=colors)
axes[0].axvline(0, color='gray', lw=0.8, ls='--')
axes[0].set_title('LIME\n(local linear weights)', fontsize=10)
axes[0].set_xlabel('Weight')

# — SHAP (middle) —
top10 = np.argsort(np.abs(sv[idx]))[-10:]
shap_vals_top = sv[idx][top10]
shap_colors   = ['#3fb950' if v > 0 else '#f78166' for v in shap_vals_top]
axes[1].barh([feature_names[i] for i in top10], shap_vals_top, color=shap_colors)
axes[1].axvline(0, color='gray', lw=0.8, ls='--')
axes[1].set_title('SHAP\n(Shapley values)', fontsize=10)
axes[1].set_xlabel('SHAP value')

# — PDP (right) — show where this instance falls
fi = top4[0]
grid = np.linspace(X_test[:, fi].min(), X_test[:, fi].max(), 80)
X_cp = X_test.copy()
pdp_avg = []
for v in grid:
    X_cp[:, fi] = v
    pdp_avg.append(model.predict_proba(X_cp)[:, 1].mean())

axes[2].plot(grid, pdp_avg, color='#58a6ff', lw=2, label='PDP average')
axes[2].fill_between(grid, pdp_avg, alpha=0.1, color='#58a6ff')
axes[2].axvline(X_test[idx, fi], color='#f78166', lw=1.5, ls='--', label=f'Instance #{idx}')
axes[2].set_title(f'PDP\n({feature_names[fi]})', fontsize=10)
axes[2].set_xlabel('Feature value')
axes[2].set_ylabel('P(benign)')
axes[2].set_ylim(0, 1)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Summary

| | LIME | SHAP | PDP |
|---|---|---|---|
| **Scope** | Local | Local + Global | Global |
| **Speed** | Medium | Fast (trees) | Medium |
| **Theoretically grounded** | ✗ | ✓ | ✗ |
| **Handles feature interactions** | ✗ | ✓ | ✗ (2D PDPs help) |
| **Best for** | Explaining one prediction | Feature importance + audits | Communicating trends |

**Recommendation:** Use SHAP as your primary tool (most principled), supplement with LIME for stakeholder demos, and PDP for communicating marginal effects to non-technical audiences.